# 10 — Performance per turn, simulation and participant round

`08_study_distributions` §4 for the participant round alone, here with the
pooled simulation runs beside it: latency, model calls, input and output
tokens per agent turn. One panel per measure (two measures on different scales
never share an axis), one row per round in each panel. Every turn is a dot; the
box over it marks the median (thick bar), the quartiles (box) and the furthest
turn within 1.5 IQR (whiskers).

**Which turns.** As in `08`: every completed turn (`completed_at` set). That
includes turns that failed without a response: in the participant round these
sit at the 85 s deadline in the latency panel, and their model calls are the
ones made before the failure. They are **left out of the two token panels**: a
failed turn records 0 tokens, because the usage of its completed calls is lost
with the call that failed, and a 0 there would read as a measurement. Simulated turns are the pooled baseline and
regression runs, latest attempt of each execution only (D53), as in `09`.

Descriptive only (D104): no test, no interval.

**Privacy.** This notebook reads participant rows, as numbers only. Outputs are
stripped on commit (`notebooks/study/.gitattributes`); figures go to
`figures/study/`, which is git-ignored.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parents[1]))

import matplotlib.pyplot as plt
import pandas as pd

from src import analysis, db, figures, pilot

# --- parameters -------------------------------------------------------------
RUN_TYPE = "study"
SIM_RUN_IDS = [
    "1a872c09-7070-4789-a806-904969f9185d",  # B10 baseline run, adopted (D96)
    "e097f78e-3326-4aea-acaa-0b904d3b0a1c",  # B11 regression run
]
save = figures.Saver(RUN_TYPE, enabled=True)
figures.setup()

client = db.admin()
turns = list(pilot.load(client, RUN_TYPE).suggestions)
sim_executions, superseded = analysis.simulated_executions(client, SIM_RUN_IDS)
sim_ids = [t["id"] for ex in sim_executions for t in ex]

# simulated_executions reads only what coverage needs; the performance columns
# are fetched here for the same turns.
PERF = "id, completed_at, error_code, latency_ms, llm_calls, tokens_input, tokens_output"
sim_turns = [r for i in range(0, len(sim_ids), 100)
             for r in client.table("study_suggestions").select(PERF).in_("id", sim_ids[i:i + 100]).execute().data]
assert len(sim_turns) == len(sim_ids), "a simulated turn could not be read back"

done = {"simulation": [t for t in sim_turns if t.get("completed_at")],
        "participant": [t for t in turns if t.get("completed_at")]}
print(f"simulation: {len(done['simulation'])} completed turns ({superseded} superseded attempts left out, D53)")
print(f"participant ({RUN_TYPE}): {len(done['participant'])} completed turns")

## The numbers

Per measure and round: n, median, quartiles, 90th percentile and maximum. `n`
is lower in the token rows by the turns that failed without a response (see
above).

In [ ]:
MEASURES = [  # (column, scale, panel title, axis label, unit)
    ("latency_ms", 1000, "Latency", "seconds per turn", "s"),
    ("llm_calls", 1, "Model calls", "calls per turn", "calls"),
    ("tokens_input", 1000, "Input tokens", "thousand tokens per turn", "k tokens"),
    ("tokens_output", 1, "Output tokens", "tokens per turn", "tokens"),
]

def values(rnd, key, scale):
    keep = [t for t in done[rnd] if not (key.startswith("tokens_") and t.get("error_code"))]
    return [t[key] / scale for t in keep if t.get(key) is not None]

summary = []
for key, scale, title, _, unit in MEASURES:
    for rnd in done:
        s = pd.Series(values(rnd, key, scale), dtype=float)
        summary.append({"measure": f"{title} ({unit})", "round": rnd, "n": len(s), "median": s.median(),
                        "q1": s.quantile(0.25), "q3": s.quantile(0.75), "p90": s.quantile(0.9), "max": s.max()})
pd.DataFrame(summary).set_index(["measure", "round"]).style.format(
    {"n": "{:.0f}", **{c: "{:.1f}" for c in ("median", "q1", "q3", "p90", "max")}}, na_rep="—")

## Figure

The four panels together, then each on its own for a thesis page that has room
for one.

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(10, 4.6))
for ax, (key, scale, title, xlabel, _) in zip(axs.flat, MEASURES):
    figures.paired_distribution(ax, {rnd: values(rnd, key, scale) for rnd in done}, title, xlabel)
fig.tight_layout(); save(fig, "perf_sim_vs_participant"); plt.show()

for key, scale, title, xlabel, _ in MEASURES:
    fig, ax = plt.subplots(figsize=(6.4, 2.2))
    figures.paired_distribution(ax, {rnd: values(rnd, key, scale) for rnd in done}, title, xlabel)
    save(fig, f"perf_{key}"); plt.close(fig)
print(f"single panels written to {save.dir}/perf_<measure>.pdf|png")

## Medians

The median per measure and round, with the number of turns it is taken over:
the thick bar in each box above. The median rather than the mean, because
every measure is skewed right with a long tail (the 85 s timeouts, a few turns
with thousands of output tokens).

In [ ]:
rows = []
for key, scale, title, _, unit in MEASURES:
    row = {"measure": f"{title} ({unit})"}
    for rnd in done:
        vals = values(rnd, key, scale)
        row[f"{rnd} median"] = pd.Series(vals, dtype=float).median()
        row[f"{rnd} n"] = len(vals)
    rows.append(row)
medians = pd.DataFrame(rows).set_index("measure")
medians.style.format({c: "{:.1f}" for c in medians.columns if c.endswith("median")})